# Fabric Connection Inventory → `lh_fabric_management`

Enumerates every **connection** in the tenant and maps each to the **semantic
models** (via the metadata scanner, tenant-wide) and the **items** (via per-item
lookups, exact where accessible) that use it, then writes temporal Delta tables
to the **`fabricmanagement`** schema of the **`lh_fabric_management`** lakehouse.

## Tables written
| Table | Grain | Temporal | Contents |
|---|---|---|---|
| `connections` | one row per connection version | **SCD2** | Type, gateway, datasource path, privacy, credential type, SSO, encryption, flags |
| `connection_activity` | one row per (connection, run) | snapshot | created / last-bound / last-credential-used timestamps over time |
| `connection_semantic_model_map` | one row per (semantic model × datasource) | snapshot | Every datasource a model uses, tenant-wide, classified by `source_kind` / `risk_tier` (URL-decoded `path`); connection match is optional enrichment |
| `connection_item_map` | one row per (connection × item) | snapshot | Per-item exact; all item types; only items the run identity can read |

## How usage is built
- **Scanner (tenant-wide):** `GET /admin/workspaces/getInfo` → `scanResult` gives each dataset's `datasourceUsages` → `datasourceInstances` (with `gatewayId` + connection path). Those are matched back to `connections` on **gatewayId + path** (exact for gateway-bound, path-only for cloud).
- **Per-item (exact, partial):** admin `List Items` enumerates every item tenant-wide; `List Item Connections` is then called per item. It needs *read/write* on the item, so items the identity can't access are **skipped and counted** (there is no admin variant).

## Coverage & limitations
There is **no tenant-wide API for the connection objects themselves.** `/v1/connections` is permission-scoped, and the admin candidates — `/v2.0/myorg/gatewayClusterDatasources` and per-cluster `.../gatewayclusters/{id}/datasources` — return **HTTP 501 (Not Implemented)**. So the **`connections`** table is **caller-scoped** (every row tagged `coverage = "caller-scoped"`): it holds only connections the run identity has a role on. The **tenant-wide** signal is **`connection_semantic_model_map`** — the metadata scanner surfaces the connection *paths / gateways in use* by semantic models across every workspace, including connections you don't own. Treat that table (not `connections`) as the "what's used across the tenant" source.

## Prerequisites
1. **`lh_fabric_management` exists and is schema-enabled** (tables land in `Tables/fabricmanagement/…`). Set `LAKEHOUSE_SCHEMA = None` for a classic lakehouse.
2. Run identity is a **Fabric / Power BI Administrator** — the connections list, scanner, and admin item list all require it. (Headless SP/MI additionally needs *"Service principals can use Power BI APIs"* + the Fabric Administrator role.)
3. Per-item mapping uses `Item.ReadWrite.All`; coverage is limited to items the identity can access.

## Temporal model
`connections` is **SCD Type 2** (`valid_from`/`valid_to`/`is_current` + `row_hash` over config columns only — recency lives in `connection_activity` so it never churns the dimension). The three bridge tables are **daily snapshots** partitioned by `snapshot_date`.

In [ ]:
# ── Config & auth ────────────────────────────────────────────────────────────
from datetime import datetime, timezone
import time, json
import requests
import notebookutils            # older runtimes: use `mssparkutils` instead
import re
from urllib.parse import unquote

POWERBI_API = "https://api.powerbi.com/v1.0/myorg"
FABRIC_API  = "https://api.fabric.microsoft.com/v1"

LAKEHOUSE_NAME       = "lh_fabric_management"
LAKEHOUSE_SCHEMA     = "fabricmanagement"      # None for a classic (non-schema) lakehouse
TBL_CONNECTIONS      = "connections"
TBL_CONN_ACTIVITY    = "connection_activity"
TBL_CONN_MODEL_MAP   = "connection_semantic_model_map"
TBL_CONN_ITEM_MAP    = "connection_item_map"

EXCLUDE_PERSONAL = True     # skip personal workspaces in the scanner
SCAN_BATCH       = 100      # workspaces per getInfo call
POLL_SECONDS     = 3
POLL_MAX         = 200
# Only item types that can actually bind a connection (skips reports, dashboards,
# lakehouses, SQL endpoints, etc.). Set None to probe every active item.
ITEM_TYPES       = ["SemanticModel", "Dataflow", "Datamart", "DataPipeline",
                    "Eventstream", "CopyJob", "MirroredDatabase"]
MAX_ITEMS        = None     # optional cap on per-item calls; None = no cap (logged if hit)
MAX_WORKERS      = 8        # concurrent List Item Connections calls (429 backoff self-throttles)

SCAN_TS = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
print("Scan timestamp (UTC):", SCAN_TS.isoformat())

_TOKEN = {"v": None, "exp": 0.0}
def _headers():
    # Cache the bearer token so a parallel/large loop doesn't call getToken per request.
    if time.time() > _TOKEN["exp"]:
        _TOKEN["v"] = notebookutils.credentials.getToken("pbi")
        _TOKEN["exp"] = time.time() + 3000     # refresh well before the ~60-min expiry
    return {"Authorization": f"Bearer {_TOKEN['v']}", "Content-Type": "application/json"}

def _raise(resp):
    if resp.status_code in (401, 403):
        raise PermissionError(
            f"HTTP {resp.status_code} on {resp.url}\n"
            "  -> The identity needs Fabric / Power BI Administrator rights for this call.")
    resp.raise_for_status()

def _get(url, params=None):
    r = requests.get(url, headers=_headers(), params=params, timeout=120)
    _raise(r)
    return r.json()

def _post(url, params=None, body=None):
    r = requests.post(url, headers=_headers(), params=params, json=body, timeout=120)
    _raise(r)
    return r.json()

def _get_resp(url, tries=6):
    # Returns the raw Response, honoring 429 Retry-After. Caller inspects status
    # (used by the per-item loop, where 401/403/404 mean "skip this item").
    r = None
    for _ in range(tries):
        r = requests.get(url, headers=_headers(), timeout=120)
        if r.status_code == 429:
            time.sleep(min(int(r.headers.get("Retry-After", "10")), 60))
            continue
        return r
    return r

# --- data-source path classification (governance rollups) --------------------
def _readable(p):
    # URL-decode for display: "Commission%20Report" -> "Commission Report".
    return unquote(p) if p else p

def classify_path(path):
    # -> (source_kind, source_host, risk_tier)
    if not path:
        return ("Unknown", None, "Unknown")
    p = _readable(path).strip()
    low = p.lower()
    m = re.match(r"file:///([a-z]):", low)                # file:///K:/...  local/mapped drive
    if m:
        return ("LocalMappedDrive", f"{m.group(1).upper()}:", "High")
    if low.startswith("file://"):                         # file://server/share  UNC share
        return ("FileShareUNC", p[len("file://"):].split("/")[0] or None, "Medium")
    if "sharepoint.com" in low:
        host = re.sub(r"^https?://", "", p).split("/")[0]
        personal = "/personal/" in low
        return ("SharePointPersonal" if personal else "SharePoint", host, "High" if personal else "Medium")
    if ".datawarehouse.fabric.microsoft.com" in low:
        return ("FabricWarehouse", p.split(";")[0], "Low")
    if ".database.windows.net" in low:
        return ("AzureSQL", p.split(";")[0], "Low")
    if low.startswith("dsn="):
        return ("ODBC_DSN", p.split("=", 1)[1], "Medium")
    if low.startswith("http"):
        return ("Web", re.sub(r"^https?://", "", p).split("/")[0], "Medium")
    if ";" in p:                                          # other host;db (SQL, etc.)
        return ("Database", p.split(";")[0], "Low")
    return ("Other", None, "Unknown")

In [ ]:
# ── 1. Connections (tenant-wide; admin sees all) ─────────────────────────────
def list_connections():
    out, url = [], f"{FABRIC_API}/connections"
    while url:
        data = _get(url)
        out.extend(data.get("value", []))
        url = data.get("continuationUri")
    return out

connections_raw = list_connections()
print(f"Connections in tenant: {len(connections_raw)}")

from collections import Counter
for t, n in Counter(c.get("connectivityType") for c in connections_raw).most_common():
    print(f"  {str(t):<30} {n}")

# Indexes for scanner matching: (gatewayId, path) exact, and path-only fallback.
def _norm(s):
    return (s or "").strip().lower()

conn_by_id     = {c.get("id"): c for c in connections_raw}
conn_by_gwpath = {}
conn_by_path   = {}
for c in connections_raw:
    p = _norm((c.get("connectionDetails") or {}).get("path"))
    conn_by_gwpath[(_norm(c.get("gatewayId")), p)] = c
    conn_by_path.setdefault(p, []).append(c)

In [ ]:
# ── 2. Scan the tenant (getInfo -> poll -> scanResult), batched ──────────────
def modified_workspace_ids():
    data = _get(f"{POWERBI_API}/admin/workspaces/modified",
                params={"excludePersonalWorkspaces": str(EXCLUDE_PERSONAL).lower()})
    items = data if isinstance(data, list) else data.get("value", [])
    return [w["id"] for w in items]

def scan_workspaces(ws_ids):
    workspaces, instances = [], {}
    for start in range(0, len(ws_ids), SCAN_BATCH):
        batch = ws_ids[start:start + SCAN_BATCH]
        scan = _post(f"{POWERBI_API}/admin/workspaces/getInfo",
                     params={"lineage": "true", "datasourceDetails": "true",
                             "datasetSchema": "false", "datasetExpressions": "false",
                             "getArtifactUsers": "false"},
                     body={"workspaces": batch})
        scan_id = scan["id"]
        for _ in range(POLL_MAX):
            status = _get(f"{POWERBI_API}/admin/workspaces/scanStatus/{scan_id}").get("status")
            if status == "Succeeded":
                break
            if status in ("Failed", "Disabled"):
                raise RuntimeError(f"Scan {scan_id} ended with status={status}")
            time.sleep(POLL_SECONDS)
        else:
            raise TimeoutError(f"Scan {scan_id} did not finish after {POLL_MAX*POLL_SECONDS}s")
        result = _get(f"{POWERBI_API}/admin/workspaces/scanResult/{scan_id}")
        workspaces.extend(result.get("workspaces", []) or [])
        for inst in (result.get("datasourceInstances") or []):
            instances[inst.get("datasourceId")] = inst
    return workspaces, instances

ws_ids = modified_workspace_ids()
print(f"Workspaces to scan: {len(ws_ids)}")
scanned_workspaces, datasource_instances = scan_workspaces(ws_ids)
ws_name = {w.get("id"): w.get("name") for w in scanned_workspaces}
print(f"Datasource instances (deduped): {len(datasource_instances)}")

In [ ]:
# ── 3. Per-item connections (parallel; exact where the identity has access) ──
from concurrent.futures import ThreadPoolExecutor, as_completed

def list_admin_items(types):
    items = []
    for t in (types or [None]):
        url = f"{FABRIC_API}/admin/items" + (f"?type={t}" if t else "")
        while url:
            data = _get(url)
            items.extend(data.get("itemEntities", []))
            url = data.get("continuationUri")
    return items

def item_connections(ws, item):
    # Returns list of connections, or None if the item is inaccessible/unsupported.
    out, url = [], f"{FABRIC_API}/workspaces/{ws}/items/{item}/connections"
    while url:
        r = _get_resp(url)
        if r.status_code in (401, 403, 404):
            return None
        r.raise_for_status()
        data = r.json()
        out.extend(data.get("value", []))
        url = data.get("continuationUri")
    return out

def _probe(it):
    try:
        return it, item_connections(it.get("workspaceId"), it.get("id")), None
    except Exception as e:
        return it, None, str(e)          # one bad item must not kill the whole run

items = list_admin_items(ITEM_TYPES)
if MAX_ITEMS and len(items) > MAX_ITEMS:
    print(f"NOTE: capping per-item calls at {MAX_ITEMS} of {len(items)} items (MAX_ITEMS set).")
    items = sorted(items, key=lambda i: i.get("id", ""))[:MAX_ITEMS]
print(f"Items to probe for connections: {len(items)}  (workers={MAX_WORKERS})")

# Fan the independent per-item calls out across a thread pool; build rows in the
# main thread as futures complete (so conn_item_rows needs no lock).
conn_item_rows = []
probed = skipped = with_conns = failed = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for fut in as_completed([ex.submit(_probe, it) for it in items]):
        it, res, err = fut.result()
        probed += 1
        if err is not None:
            failed += 1
            continue
        if res is None:
            skipped += 1
            continue
        if res:
            with_conns += 1
        ws = it.get("workspaceId")
        for ic in res:
            cd = ic.get("connectionDetails") or {}
            _sk, _sh, _rt = classify_path(cd.get("path"))
            conn_item_rows.append({
                "connection_id": ic.get("id"),
                "connection_name": ic.get("displayName") or (conn_by_id.get(ic.get("id")) or {}).get("displayName"),
                "connectivity_type": ic.get("connectivityType"),
                "path": _readable(cd.get("path")),
                "source_kind": _sk,
                "source_host": _sh,
                "risk_tier": _rt,
                "workspace_id": ws,
                "workspace_name": ws_name.get(ws),
                "item_id": it.get("id"),
                "item_name": it.get("name"),
                "item_type": it.get("type"),
                "scan_timestamp": SCAN_TS,
            })
        if probed % 200 == 0:
            print(f"  probed {probed}/{len(items)} (skipped {skipped}, with-conn {with_conns}, failed {failed})")

print(f"Per-item: probed {probed}, with-connections {with_conns}, "
      f"skipped-no-access {skipped}, failed {failed}, link rows {len(conn_item_rows)}")

In [ ]:
# ── 4. Build the remaining rows ──────────────────────────────────────────────
def _conn_path(inst):
    cd = inst.get("connectionDetails") or {}
    if isinstance(cd, str):
        try:
            cd = json.loads(cd)
        except Exception:
            cd = {}
    server, db = cd.get("server"), cd.get("database")
    if server and db:
        return f"{server};{db}"
    return server or cd.get("url") or cd.get("path") or ""

def match_connection(inst):
    gw, p = _norm(inst.get("gatewayId")), _norm(_conn_path(inst))
    if (gw, p) in conn_by_gwpath:
        return conn_by_gwpath[(gw, p)], "gateway+path"
    cands = conn_by_path.get(p, [])
    if len(cands) == 1:
        return cands[0], "path-only"
    if len(cands) > 1:
        return None, "ambiguous-path"
    return None, "no-match"

# 4a. connections dimension (config attributes only — no recency here) ---------
connection_rows = []
for c in connections_raw:
    cd = c.get("connectionDetails") or {}
    cr = c.get("credentialDetails") or {}
    rec = c.get("connectionRecency") or {}
    _sk, _sh, _rt = classify_path(cd.get("path"))
    connection_rows.append({
        "connection_id": c.get("id"),
        "display_name": c.get("displayName"),
        "connectivity_type": c.get("connectivityType"),
        "gateway_id": c.get("gatewayId"),
        "datasource_type": cd.get("type"),
        "path": _readable(cd.get("path")),
        "privacy_level": c.get("privacyLevel"),
        "credential_type": cr.get("credentialType"),
        "single_sign_on_type": cr.get("singleSignOnType"),
        "connection_encryption": cr.get("connectionEncryption"),
        "skip_test_connection": cr.get("skipTestConnection"),
        "allow_usage_in_gateway": c.get("allowConnectionUsageInGateway"),
        "allow_usage_in_user_code": c.get("allowUsageInUserControlledCode"),
        "created_datetime": rec.get("createdDateTime"),
        "coverage": "caller-scoped",   # no tenant-wide connections API (v2.0 admin = HTTP 501)
        "source_kind": _sk,
        "source_host": _sh,
        "risk_tier": _rt,
    })

# 4b. activity snapshot (recency series) ---------------------------------------
activity_rows = []
for c in connections_raw:
    rec = c.get("connectionRecency") or {}
    activity_rows.append({
        "connection_id": c.get("id"),
        "created_datetime": rec.get("createdDateTime"),
        "last_bound_datetime": rec.get("lastBoundDateTime"),
        "last_credential_used_datetime": rec.get("lastCredentialUsedDateTime"),
        "scan_timestamp": SCAN_TS,
    })

# 4c. semantic model <-> datasource (scanner; tenant-wide, classified) ---------
# Emit EVERY datasource a model uses across the tenant -- not just ones matched
# to a connection you own -- so file-share / mapped-drive sources are captured.
# The connection match is optional enrichment (connection_* may be null).
conn_model_rows = []
match_stats = {}
for wsp in scanned_workspaces:
    wsid, wsn = wsp.get("id"), wsp.get("name")
    for ds in (wsp.get("datasets") or []):
        usages = (ds.get("datasourceUsages") or []) + (ds.get("misconfiguredDatasourceUsages") or [])
        seen = set()
        for u in usages:
            iid = u.get("datasourceInstanceId")
            if iid in seen:
                continue
            seen.add(iid)
            inst = datasource_instances.get(iid)
            if not inst:
                continue
            conn, how = match_connection(inst)          # optional enrichment
            match_stats[how] = match_stats.get(how, 0) + 1
            path = _conn_path(inst)
            sk, sh, rt = classify_path(path)
            conn_model_rows.append({
                "datasource_type": inst.get("datasourceType"),
                "path": _readable(path),
                "source_kind": sk,
                "source_host": sh,
                "risk_tier": rt,
                "gateway_id": inst.get("gatewayId"),
                "connection_id": conn.get("id") if conn else None,
                "connection_name": conn.get("displayName") if conn else None,
                "connectivity_type": conn.get("connectivityType") if conn else None,
                "match_method": how,
                "workspace_id": wsid,
                "workspace_name": wsn,
                "semantic_model_id": ds.get("id"),
                "semantic_model_name": ds.get("name"),
                "scan_timestamp": SCAN_TS,
            })

print(f"connections ....................... {len(connection_rows)}")
print(f"model <-> datasource (scanner) .... {len(conn_model_rows)}  (conn match: {match_stats})")
print(f"connection <-> item (per-item) .... {len(conn_item_rows)}")

In [ ]:
# ── 5. Write to lh_fabric_management (SCD2 dim + daily-snapshot bridges) ──────
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType, TimestampType)

SNAP_DATE = SCAN_TS.date()

def _rows(dicts, schema):
    names = [f.name for f in schema.fields]
    return [tuple(d.get(n) for n in names) for d in dicts]

def _tables_path(lakehouse_name):
    lh = notebookutils.lakehouse.get(lakehouse_name)
    props = (lh.get("properties") or {}) if isinstance(lh, dict) else {}
    return (props.get("oneLakeTablesPath")
            or f'{props.get("abfsPath", "").rstrip("/")}/Tables')

TABLES_PATH = _tables_path(LAKEHOUSE_NAME)

def _table_uri(name):
    sub = name if not LAKEHOUSE_SCHEMA else f"{LAKEHOUSE_SCHEMA}/{name}"
    return f"{TABLES_PATH}/{sub}"

def _df(dicts, schema):
    return spark.createDataFrame(_rows(dicts, schema), schema=schema)

def _with_hash(df, cols):
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in cols]
    return df.withColumn("row_hash", F.sha2(F.concat_ws("||", *parts), 256))

def write_scd2(dicts, schema, name, keys):
    path = _table_uri(name)
    biz = [f.name for f in schema.fields]
    src = _with_hash(_df(dicts, schema), biz)
    if not DeltaTable.isDeltaTable(spark, path):
        (src.withColumn("valid_from", F.lit(SCAN_TS).cast("timestamp"))
            .withColumn("valid_to",  F.lit(None).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .write.format("delta").save(path))
        print(f"  {src.count():>5} rows -> {name} (initial SCD2 load)")
        return
    dt = DeltaTable.forPath(spark, path)
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    (dt.alias("t").merge(src.alias("s"), f"({cond}) AND t.is_current = true")
       .whenMatchedUpdate(condition="t.row_hash <> s.row_hash",
                          set={"is_current": F.lit(False), "valid_to": F.lit(SCAN_TS)})
       .whenNotMatchedBySourceUpdate(condition="t.is_current = true",
                          set={"is_current": F.lit(False), "valid_to": F.lit(SCAN_TS)})
       .execute())
    cur = dt.toDF().where("is_current = true").select(*keys)
    ins = src.join(cur, keys, "left_anti")
    (ins.withColumn("valid_from", F.lit(SCAN_TS).cast("timestamp"))
        .withColumn("valid_to",  F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").mode("append").save(path))
    print(f"  +{ins.count()} new/changed versions -> {name} (SCD2 merge)")

def write_snapshot(dicts, schema, name):
    path = _table_uri(name)
    df = _df(dicts, schema).withColumn("snapshot_date", F.lit(SNAP_DATE))
    w = (df.write.format("delta").partitionBy("snapshot_date")
           .option("overwriteSchema", "true").mode("overwrite"))
    if DeltaTable.isDeltaTable(spark, path):
        w = w.option("replaceWhere", f"snapshot_date = '{SNAP_DATE}'")
    w.save(path)
    print(f"  {df.count():>5} rows -> {name} (snapshot {SNAP_DATE})")

connections_schema = StructType([
    StructField("connection_id", StringType()),
    StructField("display_name", StringType()),
    StructField("connectivity_type", StringType()),
    StructField("gateway_id", StringType()),
    StructField("datasource_type", StringType()),
    StructField("path", StringType()),
    StructField("privacy_level", StringType()),
    StructField("credential_type", StringType()),
    StructField("single_sign_on_type", StringType()),
    StructField("connection_encryption", StringType()),
    StructField("skip_test_connection", BooleanType()),
    StructField("allow_usage_in_gateway", BooleanType()),
    StructField("allow_usage_in_user_code", BooleanType()),
    StructField("created_datetime", StringType()),
    StructField("coverage", StringType()),
    StructField("source_kind", StringType()),
    StructField("source_host", StringType()),
    StructField("risk_tier", StringType()),
])

activity_schema = StructType([
    StructField("connection_id", StringType()),
    StructField("created_datetime", StringType()),
    StructField("last_bound_datetime", StringType()),
    StructField("last_credential_used_datetime", StringType()),
    StructField("scan_timestamp", TimestampType()),
])

model_map_schema = StructType([
    StructField("datasource_type", StringType()),
    StructField("path", StringType()),
    StructField("source_kind", StringType()),
    StructField("source_host", StringType()),
    StructField("risk_tier", StringType()),
    StructField("gateway_id", StringType()),
    StructField("connection_id", StringType()),
    StructField("connection_name", StringType()),
    StructField("connectivity_type", StringType()),
    StructField("match_method", StringType()),
    StructField("workspace_id", StringType()),
    StructField("workspace_name", StringType()),
    StructField("semantic_model_id", StringType()),
    StructField("semantic_model_name", StringType()),
    StructField("scan_timestamp", TimestampType()),
])

item_map_schema = StructType([
    StructField("connection_id", StringType()),
    StructField("connection_name", StringType()),
    StructField("connectivity_type", StringType()),
    StructField("path", StringType()),
    StructField("source_kind", StringType()),
    StructField("source_host", StringType()),
    StructField("risk_tier", StringType()),
    StructField("workspace_id", StringType()),
    StructField("workspace_name", StringType()),
    StructField("item_id", StringType()),
    StructField("item_name", StringType()),
    StructField("item_type", StringType()),
    StructField("scan_timestamp", TimestampType()),
])

print(f"Writing to {LAKEHOUSE_NAME} (schema={LAKEHOUSE_SCHEMA}): {TABLES_PATH}")
write_scd2(connection_rows, connections_schema, TBL_CONNECTIONS, ["connection_id"])   # SCD2 dim
write_snapshot(activity_rows,   activity_schema,  TBL_CONN_ACTIVITY)                   # daily snapshot
write_snapshot(conn_model_rows, model_map_schema, TBL_CONN_MODEL_MAP)
write_snapshot(conn_item_rows,  item_map_schema,  TBL_CONN_ITEM_MAP)

## Verify

Tenant-wide model data sources by **risk tier / source kind**, the most-depended-on
fragile file hosts, and overall coverage.

In [ ]:
spark.read.format("delta").load(_table_uri(TBL_CONNECTIONS)).createOrReplaceTempView("connections_scd2")
spark.read.format("delta").load(_table_uri(TBL_CONN_MODEL_MAP)).createOrReplaceTempView("cmm")
spark.read.format("delta").load(_table_uri(TBL_CONN_ITEM_MAP)).createOrReplaceTempView("cim")

print("Model data sources by risk tier / source kind (latest scan):")
display(spark.sql(
    "SELECT risk_tier, source_kind, "
    "       COUNT(DISTINCT semantic_model_id) AS models, "
    "       COUNT(DISTINCT source_host)       AS hosts, "
    "       COUNT(*)                          AS datasource_links "
    "FROM cmm WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM cmm) "
    "GROUP BY risk_tier, source_kind "
    "ORDER BY CASE risk_tier WHEN 'High' THEN 0 WHEN 'Medium' THEN 1 "
    "                        WHEN 'Low' THEN 2 ELSE 3 END, models DESC"))

print("Most-depended-on fragile file hosts (mapped drives / shares / personal OneDrive):")
display(spark.sql(
    "SELECT source_kind, source_host, COUNT(DISTINCT semantic_model_id) AS models "
    "FROM cmm WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM cmm) "
    "  AND source_kind IN ('LocalMappedDrive','FileShareUNC','SharePointPersonal') "
    "GROUP BY source_kind, source_host ORDER BY models DESC LIMIT 20"))

print("Coverage:")
display(spark.sql(
    "SELECT "
    "  (SELECT COUNT(DISTINCT semantic_model_id) FROM cmm WHERE snapshot_date=(SELECT MAX(snapshot_date) FROM cmm)) AS models_scanned, "
    "  (SELECT COUNT(DISTINCT item_id) FROM cim WHERE snapshot_date=(SELECT MAX(snapshot_date) FROM cim)) AS items_with_conns, "
    "  (SELECT COUNT(*) FROM connections_scd2 WHERE is_current=true) AS connections_caller_scoped"))